In [36]:
# Cell 1: Import and connect
import psycopg2
import pandas as pd
import yaml

with open('../configs/config.yml', 'r') as f:
    config = yaml.safe_load(f)

db = config['database']
conn = psycopg2.connect(
    host=db['host'], port=db['port'],
    database=db['database'], user=db['user'], password=db['password']
)
conn.autocommit = True  # Fix for transaction issues
print("Connected")

Connected


In [37]:
# Cell 2: Overall summary
cur = conn.cursor()

summary = {}

cur.execute("SELECT COUNT(*) FROM requests")
summary['total_requests'] = cur.fetchone()[0]

statuses = ['NEW', 'LOCKED', 'FETCHED', 'PREPROCESSED', 'PREDICTED_MODEL', 'FINALIZED', 'RESPONDED', 'FAILED']
for status in statuses:
    cur.execute("SELECT COUNT(*) FROM requests WHERE status = %s", (status,))
    count = cur.fetchone()[0]
    summary[f'{status.lower()}_requests'] = count

cur.execute("SELECT COUNT(*) FROM raw_comments")
summary['total_raw_comments'] = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM preprocessed_comments")
summary['total_preprocessed'] = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM predictions")
summary['total_predictions'] = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM final_results")
summary['total_finalized'] = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM review_queue WHERE status = 'PENDING'")
summary['pending_review'] = cur.fetchone()[0]

print("SUMMARY STATISTICS")
for key, value in summary.items():
    print(f"  {key.replace('_', ' ').title()}: {value}")

SUMMARY STATISTICS
  Total Requests: 1
  New Requests: 0
  Locked Requests: 0
  Fetched Requests: 0
  Preprocessed Requests: 0
  Predicted Model Requests: 1
  Finalized Requests: 0
  Responded Requests: 0
  Failed Requests: 0
  Total Raw Comments: 127
  Total Preprocessed: 76
  Total Predictions: 76
  Total Finalized: 0
  Pending Review: 0


In [38]:

# Cell 3: Create DataFrame from summary
df_summary = pd.DataFrame([summary])
df_summary

,total_requests,new_requests,locked_requests,fetched_requests,preprocessed_requests,predicted_model_requests,finalized_requests,responded_requests,failed_requests,total_raw_comments,total_preprocessed,total_predictions,total_finalized,pending_review
0,1,0,0,0,0,1,0,0,0,127,76,76,0,0


In [40]:
# Cell 2: View predictions with original text
df_pred = pd.read_sql("""
    SELECT 
        p.req_id,
        rc.text_display as original_text,
        p.model_prediction,
        p.model_confidence,
        p.created_at
    FROM predictions p
    JOIN raw_comments rc ON p.comment_id = rc.comment_id
    ORDER BY p.created_at DESC
    LIMIT 20
""", conn)

print(f"Total predictions: {len(df_pred)}")
df_pred.head(10)

Total predictions: 20


/tmp/ipykernel_19282/2631078549.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pred = pd.read_sql("""


,req_id,original_text,model_prediction,model_confidence,created_at
0,1,ဘဝမြင့်ရောက်မှရမယ်,Sadness,0.667227,2026-05-23 22:40:36.009716
1,1,D လို music type မျိုး ဘယ်လို ခေါ်လဲ ဗျ ကြိုက်...,Surprise,0.395100,2026-05-23 22:40:35.952810
2,1,ငကျားကစောက်ပေါကွာ,Fear,0.980605,2026-05-23 22:40:35.867904
3,1,1:36ကဘယ်သူတီးထားတာလဲဟ အကျွန်ကြီး😮,Sadness,0.647063,2026-05-23 22:40:35.815365
4,1,စောက်ပေါသီချင်းကိုစောက်ပေါတေအများပြားကြိုက်တာတ...,Neutral,0.520602,2026-05-23 22:40:35.756739
5,1,Post punk vibe ကိုရတယ်,Joy,0.869699,2026-05-23 22:40:35.678017
6,1,အလွင့်ကြီး,Joy,0.769952,2026-05-23 22:40:35.626090
7,1,လိုင်းကားစီးတာနဲ့သီချင်းတစ်ပုဒ်ဖြစ်သွားတာပဲနော...,Neutral,0.511686,2026-05-23 22:40:35.574925
8,1,စုတ်ပျက်နေတာ ထပ်‌ေနတဲ့lyricsနဲ့ ဘယ်နားကောင်းတာလဲ,Neutral,0.693560,2026-05-23 22:40:35.419997
9,1,ခိုက်,Fear,0.886830,2026-05-23 22:40:35.355157


In [41]:
# Cell 2: Count predictions by request
pred_counts = pd.read_sql("""
    SELECT req_id, COUNT(*) as prediction_count
    FROM predictions
    GROUP BY req_id
    ORDER BY req_id DESC
""", conn)

print("PREDICTIONS PER REQUEST")
pred_counts

PREDICTIONS PER REQUEST


/tmp/ipykernel_19282/35150490.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pred_counts = pd.read_sql("""


,req_id,prediction_count
0,1,76


In [ ]:
# Cell 4: Close connection
cur.close()
conn.close()
print("Connection closed")